# JOTATIGER.FIT — Servidor Lip-sync (LatentSync)

Genera vídeo de Juan Fit hablando con boca sincronizada al audio.

**Pasos:**
1. Ejecuta todas las celdas en orden
2. Sube `juan-fit-avatar.jpg` cuando se pida
3. Copia la URL de ngrok en tu `.env` como `COLAB_LIPSYNC_URL`

⚠️ Requiere **GPU T4**: Entorno de ejecución → Cambiar tipo → T4

In [ ]:
# CELDA 1 — Clonar LatentSync e instalar dependencias
!git clone https://github.com/bytedance/LatentSync.git /content/LatentSync -q
%cd /content/LatentSync
!pip install -r requirements.txt -q
!pip install flask pyngrok -q
print('Instalación completada')

In [ ]:
# CELDA 2 — Descargar modelos de LatentSync
import os

!mkdir -p /content/LatentSync/checkpoints

# Descargar pesos del modelo
!wget -q https://huggingface.co/ByteDance/LatentSync/resolve/main/latentsync_unet.pt \
    -O /content/LatentSync/checkpoints/latentsync_unet.pt

!wget -q https://huggingface.co/ByteDance/LatentSync/resolve/main/whisper/tiny.pt \
    -O /content/LatentSync/checkpoints/whisper/tiny.pt 2>/dev/null || \
    (mkdir -p /content/LatentSync/checkpoints/whisper && \
     wget -q https://openaipublic.azureedge.net/main/whisper/models/65147644a518d12f04e32d6f3b26facc3f8dd46e47f8645f7f1acf82bb9ec840/tiny.pt \
     -O /content/LatentSync/checkpoints/whisper/tiny.pt)

print('Modelos descargados')

In [ ]:
# CELDA 3 — Subir imagen del avatar Juan Fit
from google.colab import files
import shutil

print('Sube el archivo juan-fit-avatar.jpg')
uploaded = files.upload()

avatar_path = '/content/juan-fit-avatar.jpg'
for filename in uploaded.keys():
    shutil.move(filename, avatar_path)
    print(f'Avatar guardado en: {avatar_path}')

In [ ]:
# CELDA 4 — Configurar ngrok
NGROK_TOKEN = "PON_TU_TOKEN_NGROK_AQUI"  # <-- cambia esto

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# CELDA 5 — Servidor Flask para lip-sync
import threading
import uuid
import subprocess
from flask import Flask, request, jsonify, send_file

app = Flask(__name__)

LATENTSYNC_DIR = '/content/LatentSync'
AVATAR_PATH = '/content/juan-fit-avatar.jpg'

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'latentsync', 'avatar': AVATAR_PATH})

@app.route('/lipsync', methods=['POST'])
def lipsync():
    try:
        # Recibe el audio como archivo
        if 'audio' not in request.files:
            return jsonify({'error': 'falta campo audio'}), 400

        audio_file = request.files['audio']
        job_id = uuid.uuid4().hex[:8]
        audio_path = f'/content/input_audio_{job_id}.wav'
        output_path = f'/content/output_{job_id}.mp4'

        audio_file.save(audio_path)

        # Ejecutar LatentSync
        cmd = [
            'python', f'{LATENTSYNC_DIR}/inference.py',
            '--unet_config_path', f'{LATENTSYNC_DIR}/configs/unet/second_stage.yaml',
            '--inference_ckpt_path', f'{LATENTSYNC_DIR}/checkpoints/latentsync_unet.pt',
            '--guidance_scale', '2.0',
            '--video_path', AVATAR_PATH,
            '--audio_path', audio_path,
            '--video_out_path', output_path
        ]

        result = subprocess.run(cmd, capture_output=True, text=True, cwd=LATENTSYNC_DIR)

        if result.returncode != 0:
            return jsonify({'error': result.stderr[-500:]}), 500

        return send_file(output_path, mimetype='video/mp4')

    except Exception as e:
        return jsonify({'error': str(e)}), 500

def run_server():
    app.run(port=5001, debug=False, use_reloader=False)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

import time
time.sleep(2)
tunnel = ngrok.connect(5001)

print('=' * 50)
print('SERVIDOR LIP-SYNC ACTIVO')
print('=' * 50)
print(f'URL pública: {tunnel.public_url}')
print(f'Endpoint: {tunnel.public_url}/lipsync')
print('=' * 50)
print('Copia esta URL en tu .env como COLAB_LIPSYNC_URL')

## Test manual
Sube un archivo .wav de prueba para verificar que el lip-sync funciona.

In [ ]:
# CELDA 6 — Test del servidor (opcional)
# Necesitas tener un audio .wav de prueba
from google.colab import files
import requests

print('Sube un archivo .wav para probar el lip-sync')
uploaded = files.upload()

for filename in uploaded.keys():
    with open(filename, 'rb') as f:
        response = requests.post(
            f'{tunnel.public_url}/lipsync',
            files={'audio': (filename, f, 'audio/wav')}
        )

    if response.status_code == 200:
        with open('/content/test_lipsync_output.mp4', 'wb') as out:
            out.write(response.content)
        print('Test exitoso. Descarga /content/test_lipsync_output.mp4')
        files.download('/content/test_lipsync_output.mp4')
    else:
        print('Error:', response.text)